# 🎓 Tutorial 3: Clinical Phenotyping & Risk Scoring

Welcome! In clinical data science, Electronic Health Record (EHR) data is messy. You will rarely find a clean column that says `has_diabetes = True` or `infection_source = 'Urinary'`.

Our goal today is to compute the **INCREMENT-ESBL Risk Score**. To do this, we must first build "Phenotypes" (clean, standardized variables) out of raw hospital data, and then feed those phenotypes into our scoring math.

In [4]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add the project root to the path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.utils import get_latest_data_dir

# 1. Load the latest synthetic raw data
latest_data_dir = get_latest_data_dir()
df_static = pd.read_csv(latest_data_dir / 'df_static.csv')
df_ts = pd.read_csv(latest_data_dir / 'df_ts_missing.csv')

# Merge them for this tutorial
df = pd.merge(df_ts, df_static, on='patient_id', how='left')
print(f"Data loaded! Shape: {df.shape}")
display(df.head(3))

Data loaded! Shape: (6000, 60)


,patient_id,step,date,onset_step,is_sick,hr,rr,o2sat,temp,sbp,...,prior_esbl_history,hosp_last_90d,abx_last_90d,hosp_last_1y,prior_abx_90d,diabetes_icd,hypertension_icd,malignancy_icd,risk,sepsis_case
0,101,0,2024-01-01 00:00:00,NaN,0,52.04,52.50,51.34,NaN,51.92,...,0,0,0,0,0,NaN,I15.9,NaN,34.5,0
1,101,1,2024-01-01 04:00:00,NaN,0,60.63,58.61,52.93,42.0,53.69,...,0,0,0,0,0,NaN,I15.9,NaN,34.5,0
2,101,2,2024-01-01 08:00:00,NaN,0,62.13,59.11,53.59,NaN,NaN,...,0,0,0,0,0,NaN,I15.9,NaN,34.5,0


## Step 1: The Messy Reality

Take a look at the data above. If we need to know if the patient's bloodstream infection (BSI) has a **Urinary Source** (which gives them a lower risk score), we can't just trust one column. The doctor might have forgotten to fill it out!

Instead, we use a **Multi-Modal approach**. We check:
1. Did the doctor explicitly chart 'Urinary'?
2. Did they have a urine culture that was positive?
3. Were they billed for a UTI using an ICD-10 code?

Let's import our helper functions to search this messy data.

In [6]:
# Import the helper methods
from src.utils import check_col_bool, check_col_contains, check_col_icd10

#print("Sample of diagnosis codes:")
#display(df[['patient_id', 'diagnosis_codes']].dropna().head())

## Step 2: Exercise - Document the phenotypes

First, read the function ```has_diabetes``` in ```src/phenotypes.py```.

See how first we define in the docstring exactly what it should do before writing any code?

See how it clearly lists out all the possible ways a doctor might figure out if the patient has diabetes just by looking at a messy chart? We don't just rely on a single 'history of diabetes' checkbox, because in the real world, that checkbox is often empty!

Instead, the docstring breaks the clinical reality down into four distinct clues:
 - An explicit history flag.
 - Official ICD-10 diagnosis codes.
 - Specific medications (like insulin or metformin).
 - Abnormal lab values (like severely elevated glucose).

Once the clinical logic is written out in plain English, writing the code becomes incredibly easy.


Task: Bridging Clinical Ideals with Data Reality
1. Review the Architecture
Open the ```src/phenotypes.py``` file. Take a moment to see how the code is structured. Notice how we separate particular components (e.g., determining if a patient has diabetes, or if the infection source is urinary) from the final scores. We have to define and extract these individual components perfectly before we can accurately compute the final math.

2. Define the Logic (Ideal vs. Reality)
Before we write the Python code for the remaining components of our clinical scores, we need to map out the logic. For each remaining phenotype or score variable, write a short explanation of how it should be computed.

In your description, you must answer two questions:

1. The Clinical Ideal: In a perfect world, how would a doctor identify this? (e.g., "The doctor would look at a perfectly filled-out 'Source of Infection' form.")

2. The Data Reality: Based on the actual columns and tables we have generated in our dataset, how will we proxy or prove this? (e.g., "Since forms are often blank, we will check if bsi_source contains the word 'urinary' OR if the diagnosis_codes column contains the ICD-10 code 'N39.0'.")